# Previsão de Séries Temporais - Tarefa 2: Modelagem com Lasso

Este notebook implementa a preparação de covariáveis, a validação interna temporal (sem vazamento do futuro) e o treinamento definitivo de modelos lineares penalizados (**Lasso**) para as diferentes séries da loja (`store_total`, `FOODS` e `HOBBIES`).

In [19]:
import pandas as pd
from pathlib import Path
import sys
RAIZ = Path.cwd().resolve()  # rode o notebook a partir da raiz do repo
sys.path.insert(0, str(RAIZ / "src"))
sys.path.insert(0, str(RAIZ))

from ts_utils import BASELINES, M, SERIES, calcular_baselines, calcular_metricas, carregar_dados

from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import mean_absolute_error

import joblib

In [20]:
series = ['store_total', 'FOODS', 'HOBBIES']

In [21]:
df_treino, df_val = carregar_dados(RAIZ)

In [22]:
df_treino

,date,store_total,FOODS,HOBBIES
0,2011-01-29,4337,3239,556
1,2011-01-30,4155,3137,498
2,2011-01-31,2816,2008,415
3,2011-02-01,3051,2258,392
4,2011-02-02,2630,2032,268
...,...,...,...,...
1880,2016-03-23,3770,2554,454
1881,2016-03-24,3970,2681,474
1882,2016-03-25,4904,3421,446
1883,2016-03-26,6139,4251,610


In [23]:
df_treino.dtypes 

date           datetime64[us]
store_total             int64
FOODS                   int64
HOBBIES                 int64
dtype: object

## 1. Validação Interna e Busca de Hiperparâmetros ($\alpha$)

Para evitar o vazamento de dados (*data leakage*), calculamos as estatísticas de padronização dos preços utilizando **todo o período de treino** (até `2016-03-27`). Em seguida, separamos um conjunto de validação interna a partir de `2016-02-28` para simular o comportamento preditivo do modelo e encontrar o melhor parâmetro de regularização `alpha` para cada série separadamente.

In [24]:
df_treino_interno = df_treino[df_treino['date'] <='2016-02-28' ]
df_validacao_interno = df_treino[df_treino['date'] > '2016-02-28' ]

In [25]:
df_covariaveis = pd.read_csv('dados/covariaveis.csv')
df_covariaveis['date'] = pd.to_datetime(df_covariaveis['date'])


# Colocando os dummies
df_covariaveis = pd.get_dummies(df_covariaveis, columns= ['event_name_1','event_type_1','event_name_2','event_type_2'])

In [ ]:
melhores_alpha = {} 
medias = {}
desvios = {}
mask_treino_total = df_covariaveis['date'] <= '2016-03-27'

for serie in series:
    media_serie = df_covariaveis.loc[mask_treino_total, f'preco_{serie}'].mean()
    desvio_serie = df_covariaveis.loc[mask_treino_total, f'preco_{serie}'].std()
    
    medias[serie] = media_serie
    desvios[serie] = desvio_serie
    
    # Cria a coluna padronizada correta no dataframe de covariáveis
    df_covariaveis[f'preco_{serie}_padronizada'] = (df_covariaveis[f'preco_{serie}'] - media_serie) / desvio_serie

df_validacao_interno = df_treino[df_treino['date'] > '2016-02-28']

df_resultado_treino = pd.merge(df_treino_interno, df_covariaveis, on='date', how='inner')
df_resultado_val = pd.merge(df_validacao_interno, df_covariaveis, on='date', how='inner')

for serie in series:
    y_treino_serie = df_resultado_treino[serie]
    y_val_serie = df_resultado_val[serie]
    
    colunas_remover = ['store_total', 'FOODS', 'HOBBIES', 'date']
    X_treino_serie = df_resultado_treino.drop(columns=colunas_remover)
    X_val_serie = df_resultado_val.drop(columns=colunas_remover)
    
    alphas = [0.01, 0.1, 1, 10]
    erros = {}
    
    for alpha in alphas:
        clf = Lasso(alpha=alpha)
        clf.fit(X_treino_serie, y_treino_serie)
        
        y_pred = clf.predict(X_val_serie)
        erro_here = mean_absolute_error(y_pred, y_val_serie)
        erros[alpha] = erro_here
        
    melhor_alpha = min(erros, key=lambda x: erros[x])
    melhores_alpha[serie] = melhor_alpha
    print(f"Melhor alpha para {serie}: {melhor_alpha}")

Melhor alpha para store_total: 10
Melhor alpha para FOODS: 10
Melhor alpha para HOBBIES: 10


/home/jaimewillian/Documentos/FGV/6P/Séries Temporais/Series-Temporais-G5/.env/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.183161e+08, tolerance: 1.815e+05
  model = cd_fast.enet_coordinate_descent(
/home/jaimewillian/Documentos/FGV/6P/Séries Temporais/Series-Temporais-G5/.env/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.129380e+08, tolerance: 1.815e+05
  model = cd_fast.enet_coordinate_descent(
/home/jaimewillian/Documentos/FGV/6P/Séries Temporais/Series-Temporais-G5/.env/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:842: Con

## 2. Treinamento Definitivo e Exportação dos Modelos

Com os melhores hiperparâmetros definidos ($\alpha = 10$ para todas as séries), ajustamos o modelo **Lasso definitivo** utilizando toda a história disponível no treino. 

Para facilitar a integração, empacotamos o modelo treinado junto com as respectivas estatísticas de padronização (`média` e `desvio`) em um dicionário e salvamos os objetos em disco utilizando a biblioteca `joblib` (`.pkl`).

In [27]:
df_resultado_treino_total = pd.merge(df_treino, df_covariaveis, on='date', how='inner')

for serie in series:
    clf_definitivo = Lasso(alpha=melhores_alpha[serie])
    y_treino_total = df_resultado_treino_total[serie]
    X_treino_total = df_resultado_treino_total.drop(columns=['store_total', 'FOODS', 'HOBBIES', 'date'])
    
    clf_definitivo.fit(X_treino_total, y_treino_total)
    
    pacote_modelo = {
        'modelo': clf_definitivo,
        f'media_{serie}': medias[serie],
        f'desvio_{serie}': desvios[serie]
    }
    
    joblib.dump(pacote_modelo, f'modelo_{serie}.pkl')
    print(f"Modelo salvo para a série: {serie}")

Modelo salvo para a série: store_total
Modelo salvo para a série: FOODS
Modelo salvo para a série: HOBBIES
